In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from block_sequences_config import BLOCK_SEQUENCES

# =============================================================================
# 1. PATHS & CONFIGURATION
# =============================================================================
INPUT_DIR = Path("synthetic_extension")
OUTPUT_DIR = Path("dataset_original_only")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_COLUMNS = [
    "shunt_voltage",
    "bus_voltage_V",
    "current_mA",
    "power_mW",
    "State",
    "Attack",
]
NUMERIC_COLUMNS = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW"]
BASE_CLASSES = ["none", "Backdoor", "syn-flood"]

# =============================================================================
# 2. LOAD ALL ORIGINAL DATA PARTS
# =============================================================================
data_dict = {}

print("--- STEP 1: Loading Part Files ---")
for cat in BASE_CLASSES:
    files = sorted(INPUT_DIR.glob(f"{cat}_part_*.csv"))
    if not files:
        raise FileNotFoundError(f"No part files found for '{cat}' in {INPUT_DIR}")
    
    dfs = []
    for f in files:
        df_part = pd.read_csv(f).drop(columns=["time"], errors="ignore")
        available_cols = [c for c in MODEL_COLUMNS if c in df_part.columns]
        dfs.append(df_part[available_cols])
        
    combined_df = pd.concat(dfs, ignore_index=True)
    data_dict[cat] = combined_df
    print(f"✓ Loaded {len(files)} parts for '{cat}' -> Total rows: {len(combined_df)}")

# =============================================================================
# 3. DYNAMICALLY COMPUTE GLOBAL UNIFORM BLOCK SIZE
# =============================================================================
print("\n--- STEP 2: Computing Global Uniform Block Size ---")

total_blocks_needed = {cat: 0 for cat in BASE_CLASSES}
for node_id, seq in BLOCK_SEQUENCES.items():
    for block_type in seq[:80]:
        base_cat = "none" if block_type == "noneX2" else block_type
        if base_cat in total_blocks_needed:
            total_blocks_needed[base_cat] += 1
        else:
            raise ValueError(f"Unknown block type '{block_type}' in node {node_id}")

# Compute max possible block size per class
max_rows_per_class = {
    cat: len(data_dict[cat]) // total_blocks_needed[cat]
    for cat in BASE_CLASSES
}

# The bottleneck class dictates the uniform block size for ALL classes
UNIFORM_BLOCK_SIZE = min(max_rows_per_class.values())

print("Calculated capacities per class:")
for cat in BASE_CLASSES:
    print(f"  - {cat}: {len(data_dict[cat])} rows / {total_blocks_needed[cat]} blocks -> max {max_rows_per_class[cat]} rows/block")

print(f"\n=> GLOBAL UNIFORM BLOCK SIZE: {UNIFORM_BLOCK_SIZE} rows per block")
print(f"=> EXPECTED ROWS PER NODE:    80 blocks * {UNIFORM_BLOCK_SIZE} = {80 * UNIFORM_BLOCK_SIZE} rows")

# Slice each class into blocks of size UNIFORM_BLOCK_SIZE
prepared_blocks = {cat: [] for cat in BASE_CLASSES}

for cat in BASE_CLASSES:
    df_cat = data_dict[cat].sample(frac=1.0, random_state=42).reset_index(drop=True)
    needed_blocks = total_blocks_needed[cat]

    for i in range(needed_blocks):
        start_idx = i * UNIFORM_BLOCK_SIZE
        end_idx = start_idx + UNIFORM_BLOCK_SIZE
        block = df_cat.iloc[start_idx:end_idx].copy().reset_index(drop=True)
        prepared_blocks[cat].append(block)

# =============================================================================
# 4. ASSEMBLE DATASETS (EXACT SAME ROW COUNT PER NODE)
# =============================================================================
print("\n--- STEP 3: Assembling & Saving Node Datasets ---")

block_pointers = {cat: 0 for cat in BASE_CLASSES}

for node_id, seq in BLOCK_SEQUENCES.items():
    node_blocks = []
    sequence_80 = seq[:80]

    for block_type in sequence_80:
        base_cat = "none" if block_type == "noneX2" else block_type
        
        idx = block_pointers[base_cat]
        block_df = prepared_blocks[base_cat][idx].copy()
        block_pointers[base_cat] += 1

        # Apply 1.3x multiplier for noneX2
        if block_type == "noneX2":
            block_df[NUMERIC_COLUMNS] = block_df[NUMERIC_COLUMNS] * 1.3

        block_df["Attack"] = block_type
        block_df["data_source"] = "Original"

        for col in NUMERIC_COLUMNS:
            block_df[col] = np.clip(block_df[col], a_min=0, a_max=None)

        node_blocks.append(block_df)

    final_node_df = pd.concat(node_blocks, ignore_index=True)
    filename = OUTPUT_DIR / f"Node_{node_id}_final_synthetic_dataset_with_source.csv"
    final_node_df.to_csv(filename, index=False)
    print(f"✓ Saved Node {node_id}: {len(final_node_df)} rows ({len(node_blocks)} blocks) -> {filename.name}")

print("\n" + "=" * 80)
print(f"Done. Every Node now has exactly {80 * UNIFORM_BLOCK_SIZE} rows in {OUTPUT_DIR}")

--- STEP 1: Loading Part Files ---
✓ Loaded 8 parts for 'none' -> Total rows: 14363
✓ Loaded 8 parts for 'Backdoor' -> Total rows: 21137
✓ Loaded 8 parts for 'syn-flood' -> Total rows: 13517

--- STEP 2: Computing Global Uniform Block Size ---
Calculated capacities per class:
  - none: 14363 rows / 352 blocks -> max 40 rows/block
  - Backdoor: 21137 rows / 135 blocks -> max 156 rows/block
  - syn-flood: 13517 rows / 153 blocks -> max 88 rows/block

=> GLOBAL UNIFORM BLOCK SIZE: 40 rows per block
=> EXPECTED ROWS PER NODE:    80 blocks * 40 = 3200 rows

--- STEP 3: Assembling & Saving Node Datasets ---
✓ Saved Node A: 3200 rows (80 blocks) -> Node_A_final_synthetic_dataset_with_source.csv
✓ Saved Node B: 3200 rows (80 blocks) -> Node_B_final_synthetic_dataset_with_source.csv
✓ Saved Node C: 3200 rows (80 blocks) -> Node_C_final_synthetic_dataset_with_source.csv
✓ Saved Node D: 3200 rows (80 blocks) -> Node_D_final_synthetic_dataset_with_source.csv
✓ Saved Node E: 3200 rows (80 blocks) -

In [2]:
import os
import joblib
import pandas as pd

# Configuration
nodes = ["A", "B", "C", "D", "E", "F", "G", "H"]
numeric_cols = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW"]

# Directories & Files
input_dir = "dataset_original_only"
output_dir = os.path.join(input_dir, "normalized")
scaler_path = os.path.join("models", "global_minmax_scaler.joblib")

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

# Load the existing global scaler fitted on synthetic data
print(f"Loading global scaler from: {scaler_path} ...")
scaler = joblib.load(scaler_path)

print("\n=== Processing & Normalizing Original Node Datasets ===")

for node in nodes:
    input_file = os.path.join(input_dir, f"Node_{node}_final_synthetic_dataset_with_source.csv")
    output_file = os.path.join(output_dir, f"Node_{node}_test_normalized.csv")
    
    if not os.path.exists(input_file):
        print(f"Warning: File not found -> {input_file}, skipping.")
        continue
    
    print(f"Processing {input_file} ...")
    df = pd.read_csv(input_file)
    
    # Normalize numeric features using the pre-fitted global scaler
    df_norm = df.copy()
    df_norm[numeric_cols] = scaler.transform(df[numeric_cols])
    
    # Save as Node_<Node>_test_normalized.csv
    df_norm.to_csv(output_file, index=False)
    
    print(f"✓ Node {node} saved to {output_file} (Rows: {len(df_norm)})")
    print(f"  Feature Min/Max -> Min: {df_norm[numeric_cols].min().min():.4f}, Max: {df_norm[numeric_cols].max().max():.4f}\n")

print(f"Done. All normalized original test files saved to: {output_dir}")

Loading global scaler from: models/global_minmax_scaler.joblib ...

=== Processing & Normalizing Original Node Datasets ===
Processing dataset_original_only/Node_A_final_synthetic_dataset_with_source.csv ...
✓ Node A saved to dataset_original_only/normalized/Node_A_test_normalized.csv (Rows: 3200)
  Feature Min/Max -> Min: 0.0000, Max: 0.8900

Processing dataset_original_only/Node_B_final_synthetic_dataset_with_source.csv ...
✓ Node B saved to dataset_original_only/normalized/Node_B_test_normalized.csv (Rows: 3200)
  Feature Min/Max -> Min: 0.0000, Max: 0.8648

Processing dataset_original_only/Node_C_final_synthetic_dataset_with_source.csv ...
✓ Node C saved to dataset_original_only/normalized/Node_C_test_normalized.csv (Rows: 3200)
  Feature Min/Max -> Min: 0.0000, Max: 0.9999

Processing dataset_original_only/Node_D_final_synthetic_dataset_with_source.csv ...
✓ Node D saved to dataset_original_only/normalized/Node_D_test_normalized.csv (Rows: 3200)
  Feature Min/Max -> Min: 0.0000, M